In [2]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.accuracy import rmse, mae

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
train_ratings = pd.read_csv(
    "../data/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../data/processed/test_ratings.csv"
)

movies = pd.read_csv(
    "../data/processed/movies_processed.csv"
)

print("Train:", train_ratings.shape)
print("Test:", test_ratings.shape)
print("Movies:", movies.shape)

Train: (800167, 3)
Test: (200042, 3)
Movies: (3883, 5)


In [4]:
reader = Reader(
    rating_scale=(1, 5)
)

In [5]:
train_data = Dataset.load_from_df(
    train_ratings[["userId", "movieId", "rating"]],
    reader
)

In [6]:
trainset = train_data.build_full_trainset()

In [7]:
print("Number of users:", trainset.n_users)
print("Number of movies:", trainset.n_items)
print("Number of ratings:", trainset.n_ratings)

Number of users: 6040
Number of movies: 3683
Number of ratings: 800167


In [8]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

In [9]:
svd_model.fit(trainset)

print("SVD model trained successfully!")

SVD model trained successfully!


In [10]:
testset = list(
    test_ratings[
        ["userId", "movieId", "rating"]
    ].itertuples(
        index=False,
        name=None
    )
)

In [11]:
testset[:5]

[(5412, 2683, 2.0),
 (5440, 904, 5.0),
 (368, 3717, 4.0),
 (425, 1721, 4.0),
 (4942, 3697, 1.0)]

In [12]:
predictions = svd_model.test(testset)

In [13]:
predictions[0]

Prediction(uid=5412, iid=2683, r_ui=2.0, est=np.float64(3.482833436771803), details={'was_impossible': False})

In [14]:
rmse_score = rmse(
    predictions,
    verbose=True
)

RMSE: 0.8784


In [15]:
print("SVD RMSE:", rmse_score)

SVD RMSE: 0.8783851612324959


In [16]:
mae_score = mae(
    predictions,
    verbose=True
)

print("SVD MAE:", mae_score)

MAE:  0.6902
SVD MAE: 0.6902116038110034


In [17]:
prediction_df = pd.DataFrame(
    [
        {
            "userId": pred.uid,
            "movieId": pred.iid,
            "actual_rating": pred.r_ui,
            "predicted_rating": pred.est
        }
        for pred in predictions
    ]
)

prediction_df.head()

,userId,movieId,actual_rating,predicted_rating
0,5412,2683,2.0,3.482833
1,5440,904,5.0,5.000000
2,368,3717,4.0,3.237312
3,425,1721,4.0,3.060309
4,4942,3697,1.0,2.579898


In [18]:
prediction_df["error"] = (
    prediction_df["actual_rating"]
    - prediction_df["predicted_rating"]
)

prediction_df.head()

,userId,movieId,actual_rating,predicted_rating,error
0,5412,2683,2.0,3.482833,-1.482833
1,5440,904,5.0,5.000000,0.000000
2,368,3717,4.0,3.237312,0.762688
3,425,1721,4.0,3.060309,0.939691
4,4942,3697,1.0,2.579898,-1.579898


In [21]:
prediction_df["predicted_rating"].describe()

count    200042.000000
mean          3.580629
std           0.715084
min           1.000000
25%           3.149776
50%           3.660144
75%           4.092820
max           5.000000
Name: predicted_rating, dtype: float64

In [27]:
user_rated_movies = (
    train_ratings
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("User rating history created successfully!")

User rating history created successfully!


In [28]:
def recommend_movies_svd(user_id, n=10):
    
    # Check if user exists
    if user_id not in user_rated_movies:
        return f"User {user_id} not found in the training data."
    
    # Movies already rated by this user
    rated_movies = user_rated_movies[user_id]
    
    # Movies the user has NOT rated
    candidate_movies = movies[
        ~movies["movieId"].isin(rated_movies)
    ]
    
    recommendations = []
    
    # Predict rating for every unseen movie
    for movie_id in candidate_movies["movieId"]:
        
        predicted_rating = svd_model.predict(
            user_id,
            movie_id
        ).est
        
        predicted_rating = np.clip(
            predicted_rating,
            1,
            5
        )
        
        recommendations.append(
            (movie_id, predicted_rating)
        )
    
    # Highest predicted rating first
    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )
    
    # Take top N
    top_recommendations = recommendations[:n]
    
    # Convert to DataFrame
    result = pd.DataFrame(
        top_recommendations,
        columns=["movieId", "predicted_rating"]
    )
    
    # Add movie information
    result = result.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )
    
    return result[
        ["movieId", "title", "genres", "predicted_rating"]
    ]

In [23]:
prediction_df["predicted_rating"] = np.clip(
    prediction_df["predicted_rating"],
    1,
    5
)

In [24]:
prediction_df["predicted_rating"].describe()

count    200042.000000
mean          3.580629
std           0.715084
min           1.000000
25%           3.149776
50%           3.660144
75%           4.092820
max           5.000000
Name: predicted_rating, dtype: float64

In [25]:
prediction_df["error"] = (
    prediction_df["actual_rating"]
    - prediction_df["predicted_rating"]
)


In [29]:
recommend_movies_svd(1, 10)

,movieId,title,genres,predicted_rating
0,953,It's a Wonderful Life (1946),Drama,5.000000
1,1258,"Shining, The (1980)",Horror,4.921161
2,3435,Double Indemnity (1944),Crime|Film-Noir,4.785996
3,318,"Shawshank Redemption, The (1994)",Drama,4.778384
4,1219,Psycho (1960),Horror|Thriller,4.771773
5,2905,Sanjuro (1962),Action|Adventure,4.763545
6,3038,"Face in the Crowd, A (1957)",Drama,4.763181
7,1276,Cool Hand Luke (1967),Comedy|Drama,4.751291
8,1250,"Bridge on the River Kwai, The (1957)",Drama|War,4.729721
9,3035,Mister Roberts (1955),Comedy|Drama|War,4.694250
